# Data processing
Ce notebook prépare et nettoie les données depuis la base SQLite, calcule la matrice RFM, crée le label de churn, effectue le feature engineering et sauvegarde les jeux train/val/test dans `data/processed/`.
Objectif : pipeline reproductible pour les phases ML.

In [ ]:
# Imports et configuration
import os
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta
pd.options.display.max_columns = 200
print('pandas', pd.__version__)


In [ ]:
# Définir le chemin DB (adapter si nécessaire)
DB_CANDIDATES = [
    os.path.join('..','db','retailsense.db'),
    os.path.join('..','..','db','retailsense.db'),
    os.path.join('.','db','retailsense.db')
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)
if DB_PATH is None:
    raise FileNotFoundError('retailsense.db not found; vérifiez le chemin')
conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print('Tables in DB:', tables['name'].tolist())


In [ ]:
# Chargement des tables principales (si certaines tables n'existent pas, on crée des DataFrames vides)
def safe_read(table_name):
    try:
        return pd.read_sql_query(f'SELECT * FROM {table_name}', conn)
    except Exception:
        return pd.DataFrame()

customers = safe_read('customers')
orders = safe_read('orders')
order_items = safe_read('order_items')
products = safe_read('products')
reviews = safe_read('reviews')
print('shapes -> customers, orders, order_items, products, reviews:', customers.shape, orders.shape, order_items.shape, products.shape, reviews.shape)


In [ ]:
# Nettoyage basique : casts et diagnostics
def parse_date_series(s):
    return pd.to_datetime(s, errors='coerce')

if not orders.empty:
    # colonnes courantes: order_purchase_timestamp, order_approved_at
    for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date']:
        if col in orders.columns:
            orders[col] = parse_date_series(orders[col])
    print('\norders missing summary:')
    display(orders.isnull().sum().sort_values())

if not order_items.empty:
    order_items = order_items.drop_duplicates()
    print('\norder_items missing summary:')
    display(order_items.isnull().sum().sort_values())


In [ ]:
# Jointure pour table analytique (one row per order_item). Adapter les clés si elles diffèrent.
df = order_items.copy()
if not orders.empty:
    df = df.merge(orders, on='order_id', how='left', suffixes=('_oi','_ord'))
if not customers.empty:
    df = df.merge(customers, on='customer_id', how='left')
if not products.empty and 'product_id' in df.columns:
    df = df.merge(products, on='product_id', how='left')
if not reviews.empty and 'order_id' in reviews.columns:
    df = df.merge(reviews, on='order_id', how='left')
print('Analytical DF shape:', df.shape)
display(df.head(3))


In [ ]:
# Calcul RFM par client
if orders.empty:
    raise ValueError('orders est vide : exécutez d abord db/init_db.py pour remplir la base')
reference_date = orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# s'assurer d'avoir une colonne monétaire dans orders ; sinon sommer price depuis order_items
if 'order_total' not in orders.columns:
    if 'price' in order_items.columns:
        oi_sum = order_items.groupby('order_id').agg({'price':'sum'}).rename(columns={'price':'order_total'})
        orders = orders.merge(oi_sum, on='order_id', how='left')
        orders['order_total'] = orders['order_total'].fillna(0)
    else:
        orders['order_total'] = orders.get('order_value', 0)

rfm = orders.groupby('customer_id').agg({
    'order_purchase_timestamp': lambda x: (reference_date - pd.to_datetime(x)).min(),
    'order_id': 'nunique',
    'order_total': 'sum'
}).reset_index()
rfm.columns = ['customer_id', 'recency_timedelta', 'frequency', 'monetary']
rfm['recency'] = rfm['recency_timedelta'].dt.days
rfm = rfm.drop(columns=['recency_timedelta'])
display(rfm.head())


In [ ]:
# Définition du label churn (paramétrique)
CHURN_DAYS = 180
rfm['churn_label'] = (rfm['recency'] > CHURN_DAYS).astype(int)
print('Churn distribution:')
display(rfm['churn_label'].value_counts())


In [ ]:
# Feature engineering minimal sur orders (dayofweek, month, panier total)
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'], errors='coerce')
orders['order_dow'] = orders['order_purchase_timestamp'].dt.dayofweek
orders['order_month'] = orders['order_purchase_timestamp'].dt.month
if 'order_total' not in orders.columns and 'price' in order_items.columns:
    oi_sum = order_items.groupby('order_id').agg({'price':'sum'}).rename(columns={'price':'order_total'})
    orders = orders.merge(oi_sum, on='order_id', how='left')
orders['order_total'] = orders['order_total'].fillna(0)
display(orders[['order_id','order_purchase_timestamp','order_total','order_dow']].head())


In [ ]:
# Export propre train/val/test et exemples d'exports supplémentaires (CSV)
from pathlib import Path
OUT_DIR = Path.cwd() / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def export_csv(df, name, index=False):
    path = OUT_DIR / f"{name}.csv"
    df.to_csv(path, index=index, sep=',', encoding='utf-8')
    print(f'Wrote {path} -> shape={df.shape}')

# temporal split
cut_val = orders['order_purchase_timestamp'].quantile(0.8)
cut_test = orders['order_purchase_timestamp'].quantile(0.9)
train = orders[orders['order_purchase_timestamp'] <= cut_val]
val = orders[(orders['order_purchase_timestamp'] > cut_val) & (orders['order_purchase_timestamp'] <= cut_test)]
test = orders[orders['order_purchase_timestamp'] > cut_test]

# Export : tous en CSV
export_csv(train, 'train')
export_csv(val, 'val')
export_csv(test, 'test')

# Exemples d'exports additionnels (CSV uniquement)
export_csv(rfm, 'rfm')
export_csv(orders[['order_id','customer_id','order_purchase_timestamp','order_total']], 'feature_engineering')


To run ALL in Terminal with Papermill:

& .\venv\Scripts\Activate.ps1; New-Item -ItemType Directory -Path .\notebooks\runs -Force; papermill .\notebooks\data_processing.ipynb .\notebooks\runs\data_processing_run.ipynb -k retailsense-venv --log-output
